# Trier les triptyques

Ce notebook essaie de mettre en avant les triptyques clés :
- il prends d'abord les triptyques ou il y a un personnage en sujet ou objet
- puis il regarde s'il y a un verbe dynamouv

Quel fichier triptyques filter ?

In [42]:
TRIPTYQUES_CSV_FILE = "manuel_chap1to5_chap_temps_lieux_manuels"

In [43]:
import pandas as pd

TRIPTYQUES_CSV = f"../results/csv_triptyques/{TRIPTYQUES_CSV_FILE}.csv"
DYNAMOUV_CSV = "../data/dynaMouv.csv"
OUT_CSV = f"../results/csv_triptyques/{TRIPTYQUES_CSV_FILE}_filtered.csv"

trip = pd.read_csv(TRIPTYQUES_CSV)
dyna = pd.read_csv(DYNAMOUV_CSV, sep=";")

In [44]:
verbes_dynamouv = set(dyna["Verbe"].dropna())

mask_personnage = trip["a_personnage_triptyque"]
mask_dynamouv = trip["Lemme_verbe"].fillna(trip["Verbe"]).isin(verbes_dynamouv)

#trip_filtre = trip[mask_personnage].copy()
trip_filtre = trip[mask_personnage & mask_dynamouv].copy()

trip_filtre = trip_filtre.sort_values(["chapter", "Num_paragr", "Num_phrase", "ID_verbe"],na_position="last")

trip_filtre.to_csv(OUT_CSV, index=False, encoding="utf-8")

print("Triptyques filtrés :", trip_filtre.shape)
trip_filtre.head(20)

Triptyques filtrés : (98, 52)


,Phrase,Sujet,Verbe,Objet,Dep_sujet,Dep_verbe,Dep_objet,ID_sujet,ID_verbe,ID_objet,...,time_duration_value,time_duration_unit,time_duration_relation,time_source,time_ud_governor,lieu_manuel,lieu_mention_manuelle,relation_lieu_manuelle,type_lieu_manuel,tous_lieux_manuels
43,À qui s' étonnerait de ce qu' un gentleman aus...,il,passa,sur la recommandation de MM. Baring frères,nsubj,ccomp,obl:arg,24.0,25,28.0,...,NaN,NaN,NaN,precedent_context,habitée,reform_club,cette honorable association,lieu_sans_relation,FAC,reform_club
64,Avait -il voyagé?,-il,voyagé,NaN,nsubj,root,NaN,2.0,3,NaN,...,NaN,NaN,NaN,precedent_context,habitée,NaN,NaN,NaN,NaN,NaN
66,Il n' était endroit si reculé dont il ne parût...,il,parût,dont,nsubj,acl:relcl,iobj,8.0,10,7.0,...,NaN,NaN,NaN,precedent_context,habitée,NaN,NaN,NaN,NaN,NaN
80,C' était un homme qui avait dû voyager partout...,qui,voyager,NaN,nsubj,xcomp,NaN,5.0,8,NaN,...,NaN,NaN,NaN,precedent_context,habitée,NaN,NaN,NaN,NaN,NaN
82,"Ce qui était certain toutefois, c' est que, de...",Phileas Fogg,quitté,depuis de longues années,nsubj,ccomp,obl:mod,16.0,21,14.0,...,NaN,NaN,NaN,objet,quitté,ville_londres,londres,CAD cadre,GPE,ville_londres
83,"Ce qui était certain toutefois, c' est que, de...",Phileas Fogg,quitté,Londres,nsubj,ccomp,obj,16.0,21,22.0,...,NaN,NaN,NaN,ud_direct,quitté,ville_londres,londres,CAD cadre,GPE,ville_londres
88,Ceux qui avaient l' honneur de le connaître un...,il,parcourait,qu',nsubj,acl:relcl,obj,27.0,28,26.0,...,NaN,NaN,NaN,ud_direct,parcourait,M1,qu',lieu_sans_relation,FAC,M1 ; reform_club ; maison_fogg
89,Ceux qui avaient l' honneur de le connaître un...,il,parcourait,chaque jour,nsubj,acl:relcl,obl:mod,27.0,28,30.0,...,NaN,NaN,NaN,objet,parcourait,reform_club,au club,GOL goal,FAC,reform_club ; maison_fogg ; M1
90,Ceux qui avaient l' honneur de le connaître un...,il,venir,de sa maison au club,nsubj,advcl,obl:arg,27.0,32,35.0,...,NaN,NaN,NaN,precedent_context,habitée,maison_fogg,de sa maison,SRC source,FAC,maison_fogg ; reform_club ; M1
97,"À ce jeu du silence, si bien approprié à sa na...",il,gagnait,À ce jeu du silence si bien approprié à sa nature,nsubj,root,obl:mod,14.0,15,3.0,...,NaN,NaN,NaN,precedent_context,habitée,NaN,NaN,NaN,NaN,NaN


Ajout des personnages concernées

In [ ]:
import ast
import pandas as pd
if "auto" in TRIPTYQUES_CSV_FILE :
    ENTITIES_CSV = "../data/PROPP/all_txt/tdm_auto_allchap.entities"
elif "manuel" in TRIPTYQUES_CSV_FILE:
    ENTITIES_CSV = "../data/SACR/all_annots.sacr.entities"

# Charge les entités, où il y a déjà COREF + texte de la mention
entities = pd.read_csv(ENTITIES_CSV, sep="\t")

# Garde seulement les entités personnages avec un vrai COREF
pers = entities[
    (entities["cat"] == "PER")
    & (entities["COREF"].notna())
].copy()

# COREF en texte, pour matcher avec les colonnes du fichier triptyque
pers["coref"] = pers["COREF"].astype(int).astype(str)

# On garde les mentions en nom propre, pas les pronoms
pers_noms = pers[pers["prop"] == "PROP"].copy()

# Pour chaque COREF, on prend le nom propre le plus fréquent
noms_coref = (
    pers_noms
    .value_counts(["coref", "text"])
    .reset_index(name="n")
    .sort_values(["coref", "n"], ascending=[True, False])
    .drop_duplicates("coref")
)

coref_to_nom = dict(zip(noms_coref["coref"], noms_coref["text"]))

# Lit une colonne du type "['3', '8']"
def lire_corefs(x):
    if pd.isna(x) or str(x).strip() in ["", "[]"]:
        return []
    return [str(v) for v in ast.literal_eval(str(x))]

# Transforme une liste de COREF en noms
def noms_personnages(x):
    corefs = lire_corefs(x)
    noms = [coref_to_nom.get(c, "COREF_" + c) for c in corefs]

    if len(noms) == 0:
        return pd.NA

    return "/".join(noms)

# Ajoute les noms lisibles
trip_filtre["personnage_sujet_nom"] = trip_filtre["personnages_sujet"].apply(noms_personnages)
trip_filtre["personnage_objet_nom"] = trip_filtre["personnages_objet"].apply(noms_personnages)
trip_filtre["personnages_triptyque_noms"] = trip_filtre["personnages_triptyque"].apply(noms_personnages)

trip_filtre[[
    "Sujet", "Verbe", "Objet",
    "personnages_sujet", "personnage_sujet_nom",
    "personnages_objet", "personnage_objet_nom",
    "personnages_triptyque_noms"
]].head(30)

,Sujet,Verbe,Objet,personnages_sujet,personnage_sujet_nom,personnages_objet,personnage_objet_nom,personnages_triptyque_noms
43,il,passa,sur la recommandation de MM. Baring frères,['0'],phileas fogg,['46'],baring frères,"43 phileas fogg, baring frères\n64 ..."
64,-il,voyagé,NaN,['0'],phileas fogg,[],NaN,"43 phileas fogg, baring frères\n64 ..."
66,il,parût,dont,['0'],phileas fogg,[],NaN,"43 phileas fogg, baring frères\n64 ..."
80,qui,voyager,NaN,['0'],phileas fogg,[],NaN,"43 phileas fogg, baring frères\n64 ..."
82,Phileas Fogg,quitté,depuis de longues années,['0'],phileas fogg,[],NaN,"43 phileas fogg, baring frères\n64 ..."
83,Phileas Fogg,quitté,Londres,['0'],phileas fogg,[],NaN,"43 phileas fogg, baring frères\n64 ..."
88,il,parcourait,qu',['0'],phileas fogg,[],NaN,"43 phileas fogg, baring frères\n64 ..."
89,il,parcourait,chaque jour,['0'],phileas fogg,[],NaN,"43 phileas fogg, baring frères\n64 ..."
90,il,venir,de sa maison au club,['0'],phileas fogg,['0'],phileas fogg,"43 phileas fogg, baring frères\n64 ..."
97,il,gagnait,À ce jeu du silence si bien approprié à sa nature,['0'],phileas fogg,['0'],phileas fogg,"43 phileas fogg, baring frères\n64 ..."


In [46]:
# Reprépare une table dynaMouv réduite aux colonnes utiles
dyna_type = dyna[[
    "Verbe",
    "Macro-catégorie",
    "Catégorie de base",
    "Aspect lexical",
    "Manière"
]].copy()

# Renomme les colonnes pour que ce soit plus lisible
dyna_type = dyna_type.rename(columns={
    "Verbe": "Lemme_verbe",
    "Macro-catégorie": "macro_categorie_dynamouv",
    "Catégorie de base": "categorie_base_dynamouv",
    "Aspect lexical": "aspect_lexical_dynamouv",
    "Manière": "maniere_dynamouv"
})

# Évite les doublons au moment de la jointure
dyna_type = dyna_type.drop_duplicates("Lemme_verbe")

# Ajoute les informations dynaMouv au fichier filtré
trip_presentation = trip_filtre.merge(
    dyna_type,
    on="Lemme_verbe",
    how="left"
)

# Colonne lisible avec les personnages concernés
trip_presentation["personnages_concernes"] = trip_presentation["personnages_triptyque_noms"]

# Colonnes lisibles à garder pour la présentation
colonnes_presentation = [
    "chapter",
    "Phrase",
    "Sujet",
    "Verbe",
    "Objet",
    "personnages_concernes",
    "time_code",
    "lieu_manuel",
    "type_lieu_manuel",
    "lieux_tryptiques",
    "Lemme_verbe",
    "macro_categorie_dynamouv",
    "categorie_base_dynamouv",
    "aspect_lexical_dynamouv",
    "maniere_dynamouv",
]

# Garde seulement les colonnes qui existent vraiment
colonnes_presentation = [
    c for c in colonnes_presentation
    if c in trip_presentation.columns
]

# Table finale de présentation
trip_presentation = trip_presentation[colonnes_presentation].copy()

# Export du CSV de présentation
OUT_PRESENTATION_CSV = f"../results/csv_triptyques/{TRIPTYQUES_CSV_FILE}_presentation.csv"

trip_presentation.to_csv(
    OUT_PRESENTATION_CSV,
    index=False,
    encoding="utf-8"
)

print("CSV présentation :", OUT_PRESENTATION_CSV)
print("Dimensions :", trip_presentation.shape)

trip_presentation.head(20)

CSV présentation : ../results/csv_triptyques/manuel_chap1to5_chap_temps_lieux_manuels_presentation.csv
Dimensions : (98, 14)


,chapter,Phrase,Sujet,Verbe,Objet,personnages_concernes,time_code,lieu_manuel,type_lieu_manuel,Lemme_verbe,macro_categorie_dynamouv,categorie_base_dynamouv,aspect_lexical_dynamouv,maniere_dynamouv
0,1,À qui s' étonnerait de ce qu' un gentleman aus...,il,passa,sur la recommandation de MM. Baring frères,"43 phileas fogg, baring frères\n64 ...",1872-00-00-00-00,reform_club,FAC,passer,Dpt au sens large,Dpt au sens strict,télique,NaN
1,1,Avait -il voyagé?,-il,voyagé,NaN,"43 phileas fogg, baring frères\n64 ...",1872-00-00-00-00,NaN,NaN,voyager,Dpt au sens large,Dpt au sens faible,atélique,manière
2,1,Il n' était endroit si reculé dont il ne parût...,il,parût,dont,"43 phileas fogg, baring frères\n64 ...",1872-00-00-00-00,NaN,NaN,paraître,Dpt au sens large,Dpt au sens strict,télique,NaN
3,1,C' était un homme qui avait dû voyager partout...,qui,voyager,NaN,"43 phileas fogg, baring frères\n64 ...",1872-00-00-00-00,NaN,NaN,voyager,Dpt au sens large,Dpt au sens faible,atélique,manière
4,1,"Ce qui était certain toutefois, c' est que, de...",Phileas Fogg,quitté,depuis de longues années,"43 phileas fogg, baring frères\n64 ...",1862-00-00-00-00,ville_londres,GPE,quitter,Dpt au sens large,Dpt au sens strict,télique,NaN
5,1,"Ce qui était certain toutefois, c' est que, de...",Phileas Fogg,quitté,Londres,"43 phileas fogg, baring frères\n64 ...",1862-00-00-00-00,ville_londres,GPE,quitter,Dpt au sens large,Dpt au sens strict,télique,NaN
6,1,Ceux qui avaient l' honneur de le connaître un...,il,parcourait,qu',"43 phileas fogg, baring frères\n64 ...",1872-00-00-00-00,M1,FAC,parcourir,Dpt au sens large,Dpt au sens faible,atélique,manière
7,1,Ceux qui avaient l' honneur de le connaître un...,il,parcourait,chaque jour,"43 phileas fogg, baring frères\n64 ...",1872-00-00-00-00,reform_club,FAC,parcourir,Dpt au sens large,Dpt au sens faible,atélique,manière
8,1,Ceux qui avaient l' honneur de le connaître un...,il,venir,de sa maison au club,"43 phileas fogg, baring frères\n64 ...",1872-00-00-00-00,maison_fogg,FAC,venir,Dpt au sens large,Dpt au sens strict,télique,NaN
9,1,"À ce jeu du silence, si bien approprié à sa na...",il,gagnait,À ce jeu du silence si bien approprié à sa nature,"43 phileas fogg, baring frères\n64 ...",1872-00-00-00-00,NaN,NaN,gagner,Dpt au sens large,Dpt au sens strict,télique,NaN
